# 02 — Preprocessing dan Feature Selection

**Project:** NutriSmart AI  
**Dataset utama:** `ObesityDataSet_raw_and_data_sinthetic.csv`

## Tujuan notebook

Notebook ini digunakan untuk:

1. Membaca ulang dataset mentah.
2. Menghapus baris duplikat tanpa mengubah file asli.
3. Membuat target klasifikasi biner:
   - `0 = Non-Obesity`
   - `1 = Obesity`
4. Memilih fitur utama yang benar-benar berkaitan dengan pola hidup.
5. Memisahkan fitur BMI dari fitur model AI.
6. Membagi data menjadi training dan testing secara stratified.
7. Menyusun preprocessing untuk fitur numerik dan kategorikal.
8. Memastikan preprocessing hanya belajar dari data training.
9. Menyimpan dataset bersih dan metadata untuk notebook modeling.

> **Batas notebook:** Notebook ini belum melatih Logistic Regression, Random Forest, atau Gradient Boosting. Eksperimen model dilakukan pada notebook `03_model_experiments.ipynb`.


## 1. Konsep penting sebelum mulai

### Apa itu preprocessing?

Preprocessing adalah proses menyiapkan data agar dapat digunakan oleh algoritma machine learning. Pada project ini, preprocessing mencakup:

- menghapus duplikat;
- membuat target biner;
- memilih fitur;
- memisahkan data training dan testing;
- mengisi missing value jika suatu saat muncul;
- melakukan scaling pada fitur numerik;
- melakukan encoding pada fitur kategorikal.

### Apa itu feature selection?

Feature selection adalah proses memilih kolom yang benar-benar relevan dengan tujuan model.

Tujuan model NutriSmart AI adalah:

> Menilai risiko pola hidup terhadap obesitas berdasarkan pola makan dan aktivitas fisik.

Karena itu, `Height`, `Weight`, dan BMI tidak digunakan sebagai input utama model pola hidup. Ketiganya digunakan secara terpisah untuk menampilkan kondisi tubuh pengguna saat ini.

### Apa itu data leakage?

Data leakage terjadi ketika model memperoleh informasi yang terlalu dekat dengan jawaban target. Pada dataset ini, target obesitas dibentuk berdasarkan kategori BMI. Jika `Height`, `Weight`, atau BMI ikut digunakan, model dapat menebak target dari status tubuh dan bukan dari pola hidup.


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Library berhasil dimuat.")


## 2. Menentukan lokasi project dan membaca dataset

Notebook akan mencari folder project secara otomatis sehingga tetap dapat dijalankan dari folder utama atau dari folder `notebooks/`.


In [ ]:
DATASET_FILENAME = "ObesityDataSet_raw_and_data_sinthetic.csv"

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / DATASET_FILENAME).exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Dataset tidak ditemukan. Pastikan file berada di "
        "data/raw/ObesityDataSet_raw_and_data_sinthetic.csv"
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REFERENCE_DATA_DIR = PROJECT_ROOT / "data" / "reference"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DATA_DIR / DATASET_FILENAME

print(f"Project root       : {PROJECT_ROOT}")
print(f"Dataset mentah     : {DATA_PATH}")
print(f"Folder processed   : {PROCESSED_DATA_DIR}")
print(f"Folder reference   : {REFERENCE_DATA_DIR}")
print(f"Folder artifacts   : {ARTIFACTS_DIR}")

df_raw = pd.read_csv(DATA_PATH)
display(df_raw.head())


In [ ]:
print(f"Ukuran data mentah : {df_raw.shape}")
print(f"Missing value      : {int(df_raw.isna().sum().sum())}")
print(f"Duplikat penuh     : {int(df_raw.duplicated().sum())}")


## 3. Membuat salinan dan menghapus duplikat

File dalam `data/raw/` tidak diubah. Kita membuat salinan bernama `df_clean`, lalu menghapus duplikat dari salinan tersebut.

Alasan duplikat dihapus sebelum split:

- observasi identik tidak boleh muncul di data training dan testing;
- mengurangi risiko performa model terlihat terlalu tinggi;
- menjaga evaluasi lebih jujur.


In [ ]:
df_clean = df_raw.copy()

rows_before = len(df_clean)
duplicate_count = int(df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print(f"Baris sebelum cleaning : {rows_before:,}")
print(f"Duplikat ditemukan     : {duplicate_count:,}")
print(f"Baris setelah cleaning : {rows_after:,}")
print(f"Baris yang dihapus     : {rows_removed:,}")

assert rows_removed == duplicate_count
assert df_clean.duplicated().sum() == 0


## 4. Membuat target biner

Target asli `NObeyesdad` memiliki tujuh kelas. Untuk konsep NutriSmart AI, target disederhanakan menjadi dua kelas.

### Non-Obesity (`0`)

- `Insufficient_Weight`
- `Normal_Weight`
- `Overweight_Level_I`
- `Overweight_Level_II`

### Obesity (`1`)

- `Obesity_Type_I`
- `Obesity_Type_II`
- `Obesity_Type_III`

Target baru diberi nama `Obesity_Binary`.

> Kelas `Overweight` belum dimasukkan sebagai obesitas karena target biner mengikuti pemisahan antara kategori obesitas dan non-obesitas pada label dataset.


In [ ]:
OBESITY_CLASSES = {
    "Obesity_Type_I",
    "Obesity_Type_II",
    "Obesity_Type_III",
}

df_clean["Obesity_Binary"] = (
    df_clean["NObeyesdad"].isin(OBESITY_CLASSES).astype(int)
)

target_mapping = {
    0: "Non-Obesity",
    1: "Obesity",
}

target_distribution = (
    df_clean["Obesity_Binary"]
    .value_counts()
    .sort_index()
    .rename(index=target_mapping)
    .to_frame("count")
)

target_distribution["percentage"] = (
    target_distribution["count"] / len(df_clean) * 100
).round(2)

display(target_distribution)

assert set(df_clean["Obesity_Binary"].unique()) == {0, 1}


## 5. Menentukan kelompok fitur

### Fitur utama model pola hidup

#### Pola makan

- `FAVC` — sering mengonsumsi makanan tinggi kalori
- `FCVC` — frekuensi konsumsi sayuran
- `NCP` — jumlah makanan utama
- `CAEC` — kebiasaan makan di antara waktu makan
- `CH2O` — konsumsi air harian
- `SCC` — pemantauan konsumsi kalori
- `CALC` — konsumsi alkohol

#### Aktivitas dan perilaku

- `FAF` — frekuensi aktivitas fisik
- `TUE` — waktu menggunakan perangkat teknologi
- `MTRANS` — moda transportasi

### Fitur yang tidak digunakan pada model utama

- `Height`
- `Weight`
- BMI
- `NObeyesdad`
- `Age`
- `Gender`
- `family_history_with_overweight`
- `SMOKE`

`Age`, `Gender`, dan riwayat keluarga tidak digunakan agar skor utama lebih berfokus pada pola hidup yang dapat berubah. `SMOKE` tidak digunakan karena bukan fokus utama risiko obesitas dan kategori `yes` sangat sedikit pada dataset.


In [ ]:
MAIN_NUMERIC_FEATURES = [
    "FCVC",
    "NCP",
    "CH2O",
    "FAF",
    "TUE",
]

MAIN_CATEGORICAL_FEATURES = [
    "FAVC",
    "CAEC",
    "SCC",
    "CALC",
    "MTRANS",
]

MAIN_FEATURES = MAIN_NUMERIC_FEATURES + MAIN_CATEGORICAL_FEATURES

PROFILE_FEATURES_NOT_USED = [
    "Age",
    "Gender",
    "family_history_with_overweight",
]

BODY_FEATURES_NOT_USED = [
    "Height",
    "Weight",
]

OTHER_FEATURES_NOT_USED = [
    "SMOKE",
    "NObeyesdad",
]

feature_plan = pd.DataFrame(
    {
        "feature": (
            MAIN_NUMERIC_FEATURES
            + MAIN_CATEGORICAL_FEATURES
            + PROFILE_FEATURES_NOT_USED
            + BODY_FEATURES_NOT_USED
            + OTHER_FEATURES_NOT_USED
        ),
        "group": (
            ["Main numeric lifestyle"] * len(MAIN_NUMERIC_FEATURES)
            + ["Main categorical lifestyle"] * len(MAIN_CATEGORICAL_FEATURES)
            + ["Profile - not used in main model"] * len(PROFILE_FEATURES_NOT_USED)
            + ["Body/BMI - excluded to avoid leakage"] * len(BODY_FEATURES_NOT_USED)
            + ["Other - not used"] * len(OTHER_FEATURES_NOT_USED)
        ),
    }
)

display(feature_plan)


In [ ]:
print("Distribusi SMOKE setelah cleaning:")
display(
    pd.DataFrame(
        {
            "count": df_clean["SMOKE"].value_counts(),
            "percentage": (
                df_clean["SMOKE"].value_counts(normalize=True) * 100
            ).round(2),
        }
    )
)


### Pemeriksaan anti-leakage

Cell berikut memastikan kolom kondisi tubuh dan target tidak masuk ke daftar fitur utama.


In [ ]:
forbidden_features = {
    "Height",
    "Weight",
    "BMI",
    "BMI_EDA",
    "NObeyesdad",
    "Obesity_Binary",
}

leaked_features = forbidden_features.intersection(MAIN_FEATURES)

print(f"Fitur utama: {MAIN_FEATURES}")
print(f"Fitur leakage yang ditemukan: {sorted(leaked_features)}")

assert len(leaked_features) == 0
assert len(MAIN_FEATURES) == len(set(MAIN_FEATURES))
assert set(MAIN_FEATURES).issubset(df_clean.columns)

print("Pemeriksaan anti-leakage berhasil.")


## 6. Memeriksa fitur terpilih

Sebelum split, kita memeriksa:

- tipe data;
- missing value;
- jumlah nilai unik;
- rentang fitur numerik;
- kategori fitur kategorikal.


In [ ]:
selected_feature_summary = pd.DataFrame(
    {
        "dtype": df_clean[MAIN_FEATURES].dtypes.astype(str),
        "missing": df_clean[MAIN_FEATURES].isna().sum(),
        "unique_values": df_clean[MAIN_FEATURES].nunique(),
    }
)

display(selected_feature_summary)


In [ ]:
display(df_clean[MAIN_NUMERIC_FEATURES].describe().T)


In [ ]:
for column in MAIN_CATEGORICAL_FEATURES:
    print(f"\n=== {column} ===")
    display(
        pd.DataFrame(
            {
                "count": df_clean[column].value_counts(),
                "percentage": (
                    df_clean[column].value_counts(normalize=True) * 100
                ).round(2),
            }
        )
    )


## 7. Memisahkan fitur (`X`) dan target (`y`)

- `X` berisi sepuluh fitur pola hidup.
- `y` berisi target `Obesity_Binary`.

Kolom target tidak boleh berada di dalam `X`.


In [ ]:
X = df_clean[MAIN_FEATURES].copy()
y = df_clean["Obesity_Binary"].copy()

print(f"Ukuran X : {X.shape}")
print(f"Ukuran y : {y.shape}")

display(X.head())
display(y.head().to_frame())


In [ ]:
assert "Obesity_Binary" not in X.columns
assert "NObeyesdad" not in X.columns
assert len(X) == len(y)
assert X.isna().sum().sum() == 0
assert y.isna().sum() == 0

print("Pemisahan fitur dan target berhasil.")


## 8. Membagi data training dan testing

Kita menggunakan:

- **80% data training**
- **20% data testing**
- `random_state = 42`
- `stratify = y`

### Mengapa menggunakan stratify?

Stratify menjaga proporsi kelas obesitas dan non-obesitas tetap hampir sama pada data training dan testing.

### Mengapa data testing dipisahkan sekarang?

Data testing berfungsi sebagai data yang belum pernah dilihat model. Data ini baru digunakan untuk evaluasi akhir setelah eksperimen dan pemilihan model.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")


In [ ]:
def class_distribution(series: pd.Series) -> pd.DataFrame:
    result = series.value_counts().sort_index().to_frame("count")
    result["percentage"] = (result["count"] / len(series) * 100).round(2)
    result.index = result.index.map(target_mapping)
    return result


print("Distribusi target keseluruhan:")
display(class_distribution(y))

print("Distribusi target data training:")
display(class_distribution(y_train))

print("Distribusi target data testing:")
display(class_distribution(y_test))


In [ ]:
train_indices = set(X_train.index)
test_indices = set(X_test.index)
overlap_indices = train_indices.intersection(test_indices)

print(f"Jumlah index overlap: {len(overlap_indices)}")

assert len(overlap_indices) == 0
assert len(X_train) + len(X_test) == len(X)

print("Tidak ada observasi yang muncul sekaligus pada training dan testing.")


## 9. Menyusun preprocessing pipeline

### Fitur numerik

Tahapnya:

1. `SimpleImputer(strategy="median")`
2. `StandardScaler()`

Walaupun dataset saat ini tidak memiliki missing value, imputer tetap disertakan agar pipeline lebih tahan terhadap kondisi input yang tidak sempurna pada masa depan.

`StandardScaler` mengubah skala numerik agar memiliki pusat dan penyebaran yang lebih seragam. Hal ini terutama penting untuk model seperti Logistic Regression.

### Fitur kategorikal

Tahapnya:

1. `SimpleImputer(strategy="most_frequent")`
2. `OneHotEncoder(handle_unknown="ignore")`

One-hot encoding mengubah kategori teks menjadi kolom angka 0/1. `handle_unknown="ignore"` mencegah aplikasi error ketika menerima kategori valid yang tidak muncul pada data training tertentu.


In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )
except TypeError:
    # Kompatibilitas untuk scikit-learn versi lama
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False,
    )

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", one_hot_encoder),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            MAIN_NUMERIC_FEATURES,
        ),
        (
            "categorical",
            categorical_pipeline,
            MAIN_CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print(preprocessor)


## 10. Fit preprocessing hanya pada data training

Ini merupakan aturan penting:

```text
FIT pada X_train
TRANSFORM pada X_train dan X_test
```

Kita tidak boleh melakukan `fit` pada seluruh data karena statistik data testing dapat bocor ke proses training.


In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

transformed_feature_names = preprocessor.get_feature_names_out()

print(f"Bentuk X_train sebelum preprocessing : {X_train.shape}")
print(f"Bentuk X_train setelah preprocessing : {X_train_transformed.shape}")
print(f"Bentuk X_test sebelum preprocessing  : {X_test.shape}")
print(f"Bentuk X_test setelah preprocessing  : {X_test_transformed.shape}")
print(f"Jumlah feature hasil preprocessing   : {len(transformed_feature_names)}")


In [ ]:
X_train_transformed_df = pd.DataFrame(
    X_train_transformed,
    columns=transformed_feature_names,
    index=X_train.index,
)

X_test_transformed_df = pd.DataFrame(
    X_test_transformed,
    columns=transformed_feature_names,
    index=X_test.index,
)

display(X_train_transformed_df.head())


In [ ]:
print("Nama fitur setelah preprocessing:")
for index, feature_name in enumerate(transformed_feature_names, start=1):
    print(f"{index:>2}. {feature_name}")


## 11. Validasi hasil preprocessing

Pemeriksaan meliputi:

- tidak terdapat missing value;
- tidak terdapat nilai tak hingga;
- jumlah baris tidak berubah;
- jumlah kolom training dan testing sama;
- urutan nama fitur konsisten.


In [ ]:
validation_results = {
    "train_missing": int(X_train_transformed_df.isna().sum().sum()),
    "test_missing": int(X_test_transformed_df.isna().sum().sum()),
    "train_infinite": int(
        np.isinf(X_train_transformed_df.to_numpy()).sum()
    ),
    "test_infinite": int(
        np.isinf(X_test_transformed_df.to_numpy()).sum()
    ),
    "train_rows_match": len(X_train_transformed_df) == len(X_train),
    "test_rows_match": len(X_test_transformed_df) == len(X_test),
    "same_column_count": (
        X_train_transformed_df.shape[1]
        == X_test_transformed_df.shape[1]
    ),
}

display(pd.Series(validation_results, name="value").to_frame())

assert validation_results["train_missing"] == 0
assert validation_results["test_missing"] == 0
assert validation_results["train_infinite"] == 0
assert validation_results["test_infinite"] == 0
assert validation_results["train_rows_match"]
assert validation_results["test_rows_match"]
assert validation_results["same_column_count"]

print("Semua validasi preprocessing berhasil.")


## 12. Mengapa preprocessor belum disimpan sebagai model final?

Pada notebook berikutnya, setiap algoritma akan digabungkan dengan preprocessor dalam satu `Pipeline`:

```text
Input mentah
    ↓
Preprocessing
    ↓
Model klasifikasi
    ↓
Probabilitas risiko
```

Dengan cara ini:

- tidak terjadi perbedaan preprocessing antara training dan aplikasi;
- cross-validation lebih aman;
- seluruh proses dapat disimpan menjadi satu file;
- Streamlit cukup mengirim input mentah ke pipeline final.

Preprocessor pada notebook ini hanya digunakan untuk memeriksa bahwa rancangan preprocessing bekerja dengan benar.


## 13. Menyimpan dataset bersih dan split mentah

File yang disimpan:

1. `obesity_clean_binary.csv`
2. `train_lifestyle_raw.csv`
3. `test_lifestyle_raw.csv`

Kata `raw` pada file train/test berarti fitur belum di-encoding dan belum di-scaling. Preprocessing akan tetap menjadi bagian dari pipeline model.


In [ ]:
CLEAN_DATA_PATH = PROCESSED_DATA_DIR / "obesity_clean_binary.csv"
TRAIN_DATA_PATH = PROCESSED_DATA_DIR / "train_lifestyle_raw.csv"
TEST_DATA_PATH = PROCESSED_DATA_DIR / "test_lifestyle_raw.csv"

df_clean.to_csv(CLEAN_DATA_PATH, index=False)

train_export = X_train.copy()
train_export["Obesity_Binary"] = y_train

test_export = X_test.copy()
test_export["Obesity_Binary"] = y_test

train_export.to_csv(TRAIN_DATA_PATH, index=True, index_label="source_index")
test_export.to_csv(TEST_DATA_PATH, index=True, index_label="source_index")

print(f"Dataset bersih : {CLEAN_DATA_PATH}")
print(f"Data training  : {TRAIN_DATA_PATH}")
print(f"Data testing   : {TEST_DATA_PATH}")


In [ ]:
saved_files_check = pd.DataFrame(
    [
        {
            "file": CLEAN_DATA_PATH.name,
            "exists": CLEAN_DATA_PATH.exists(),
            "size_bytes": CLEAN_DATA_PATH.stat().st_size,
        },
        {
            "file": TRAIN_DATA_PATH.name,
            "exists": TRAIN_DATA_PATH.exists(),
            "size_bytes": TRAIN_DATA_PATH.stat().st_size,
        },
        {
            "file": TEST_DATA_PATH.name,
            "exists": TEST_DATA_PATH.exists(),
            "size_bytes": TEST_DATA_PATH.stat().st_size,
        },
    ]
)

display(saved_files_check)


## 14. Menyimpan metadata preprocessing

Metadata diperlukan agar seluruh anggota tim menggunakan:

- fitur yang sama;
- target yang sama;
- pembagian data yang sama;
- random state yang sama;
- aturan preprocessing yang sama.


In [ ]:
metadata = {
    "project": "NutriSmart AI",
    "dataset_file": DATASET_FILENAME,
    "rows_raw": int(len(df_raw)),
    "duplicates_removed": int(rows_removed),
    "rows_clean": int(len(df_clean)),
    "target_source": "NObeyesdad",
    "target_name": "Obesity_Binary",
    "target_mapping": {
        "0": "Non-Obesity",
        "1": "Obesity",
    },
    "obesity_source_classes": sorted(OBESITY_CLASSES),
    "main_features": MAIN_FEATURES,
    "numeric_features": MAIN_NUMERIC_FEATURES,
    "categorical_features": MAIN_CATEGORICAL_FEATURES,
    "excluded_body_features": BODY_FEATURES_NOT_USED,
    "profile_features_not_used": PROFILE_FEATURES_NOT_USED,
    "other_features_not_used": OTHER_FEATURES_NOT_USED,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "transformed_feature_count": int(len(transformed_feature_names)),
    "transformed_feature_names": transformed_feature_names.tolist(),
    "preprocessing": {
        "numeric": [
            "SimpleImputer(strategy='median')",
            "StandardScaler()",
        ],
        "categorical": [
            "SimpleImputer(strategy='most_frequent')",
            "OneHotEncoder(handle_unknown='ignore')",
        ],
    },
}

METADATA_PATH = ARTIFACTS_DIR / "preprocessing_metadata.json"

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

print(f"Metadata disimpan ke: {METADATA_PATH}")


In [ ]:
with open(METADATA_PATH, "r", encoding="utf-8") as file:
    loaded_metadata = json.load(file)

display(pd.Series(loaded_metadata, name="value").to_frame())


## 15. Menyimpan daftar fitur sebagai referensi aplikasi

File ini belum menjadi desain input final, tetapi menjadi sumber kebenaran awal agar nama dan urutan fitur konsisten.


In [ ]:
feature_reference = pd.DataFrame(
    [
        {
            "feature": "FCVC",
            "type": "numeric",
            "minimum": 1,
            "maximum": 3,
            "description": "Frekuensi konsumsi sayuran",
        },
        {
            "feature": "NCP",
            "type": "numeric",
            "minimum": 1,
            "maximum": 4,
            "description": "Jumlah/frekuensi makanan utama",
        },
        {
            "feature": "CH2O",
            "type": "numeric",
            "minimum": 1,
            "maximum": 3,
            "description": "Konsumsi air harian",
        },
        {
            "feature": "FAF",
            "type": "numeric",
            "minimum": 0,
            "maximum": 3,
            "description": "Frekuensi aktivitas fisik",
        },
        {
            "feature": "TUE",
            "type": "numeric",
            "minimum": 0,
            "maximum": 2,
            "description": "Waktu menggunakan perangkat teknologi",
        },
        {
            "feature": "FAVC",
            "type": "categorical",
            "minimum": None,
            "maximum": None,
            "description": "Sering mengonsumsi makanan tinggi kalori",
        },
        {
            "feature": "CAEC",
            "type": "categorical",
            "minimum": None,
            "maximum": None,
            "description": "Kebiasaan makan di antara waktu makan",
        },
        {
            "feature": "SCC",
            "type": "categorical",
            "minimum": None,
            "maximum": None,
            "description": "Kebiasaan memantau konsumsi kalori",
        },
        {
            "feature": "CALC",
            "type": "categorical",
            "minimum": None,
            "maximum": None,
            "description": "Konsumsi alkohol",
        },
        {
            "feature": "MTRANS",
            "type": "categorical",
            "minimum": None,
            "maximum": None,
            "description": "Moda transportasi utama",
        },
    ]
)

FEATURE_REFERENCE_PATH = REFERENCE_DATA_DIR / "model_feature_reference.csv"
feature_reference.to_csv(FEATURE_REFERENCE_PATH, index=False)

display(feature_reference)
print(f"Referensi fitur disimpan ke: {FEATURE_REFERENCE_PATH}")


## 16. Ringkasan hasil preprocessing

Setelah notebook ini dijalankan:

1. Dataset mentah tetap aman dan tidak diubah.
2. Duplikat telah dihapus dari salinan data.
3. Target biner telah dibuat.
4. Model utama menggunakan sepuluh fitur pola hidup.
5. Tinggi, berat, dan BMI tidak digunakan sebagai fitur model.
6. Data telah dibagi 80% training dan 20% testing secara stratified.
7. Preprocessing numerik dan kategorikal telah diuji.
8. Preprocessing di-fit hanya pada data training.
9. Dataset bersih dan split mentah telah disimpan.
10. Metadata preprocessing telah disimpan.

### File hasil

- `data/processed/obesity_clean_binary.csv`
- `data/processed/train_lifestyle_raw.csv`
- `data/processed/test_lifestyle_raw.csv`
- `data/reference/model_feature_reference.csv`
- `artifacts/preprocessing_metadata.json`


In [ ]:
final_summary = {
    "raw_rows": len(df_raw),
    "clean_rows": len(df_clean),
    "duplicates_removed": rows_removed,
    "main_feature_count": len(MAIN_FEATURES),
    "numeric_feature_count": len(MAIN_NUMERIC_FEATURES),
    "categorical_feature_count": len(MAIN_CATEGORICAL_FEATURES),
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "transformed_feature_count": len(transformed_feature_names),
    "target_obesity_train": int(y_train.sum()),
    "target_non_obesity_train": int((y_train == 0).sum()),
    "target_obesity_test": int(y_test.sum()),
    "target_non_obesity_test": int((y_test == 0).sum()),
}

display(pd.Series(final_summary, name="value").to_frame())


## 17. Langkah berikutnya

Pada `03_model_experiments.ipynb`, kita akan:

1. Membaca data training dan testing hasil notebook ini.
2. Membuat pipeline untuk setiap algoritma.
3. Mencoba minimal tiga model:
   - Logistic Regression
   - Random Forest
   - Gradient Boosting
4. Melakukan cross-validation pada data training.
5. Membandingkan accuracy, precision, recall, F1-score, dan ROC-AUC.
6. Memilih kandidat model terbaik.
7. Melakukan tuning sederhana tanpa menyentuh data testing.
8. Menyimpan hasil eksperimen untuk evaluasi final.


## 18. Checklist sebelum melanjutkan

- [ ] Seluruh cell berjalan tanpa error.
- [ ] Baris duplikat berhasil dihapus dari salinan.
- [ ] Target biner berisi nilai 0 dan 1.
- [ ] `Height`, `Weight`, BMI, dan target tidak berada dalam fitur utama.
- [ ] Data training dan testing tidak memiliki index yang tumpang tindih.
- [ ] Proporsi kelas training dan testing hampir sama.
- [ ] Preprocessing di-fit hanya menggunakan `X_train`.
- [ ] Hasil preprocessing tidak memiliki missing atau infinite value.
- [ ] Lima file hasil berhasil tersimpan.
- [ ] Notebook telah disimpan dan di-commit ke Git.
